# Chiffrer et déchiffrer un message

**Hannachi — groupe B1 — travail individuel**

## Le but

Créer un petit programme qui transforme un message en texte chiffré et permet de retrouver le message avec une clé. Puis vérifier si quelqu'un peut le retrouver sans connaître cette clé.


In [1]:
def nettoyer(texte):
    resultat = ""
    for lettre in texte.upper():
        if "A" <= lettre <= "Z":
            resultat = resultat + lettre
    return resultat

def pgcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

def chiffrer(texte, a, b):
    if pgcd(a, 26) != 1:
        raise ValueError("Cle invalide : a doit etre premier avec 26")
    resultat = ""
    for lettre in nettoyer(texte):
        x = ord(lettre) - ord("A")
        y = (a * x + b) % 26
        resultat = resultat + chr(y + ord("A"))
    return resultat

def inverse(a):
    for nombre in range(26):
        if (a * nombre) % 26 == 1:
            return nombre
    raise ValueError("Cle invalide : pas d'inverse")

def dechiffrer(texte, a, b):
    a_inverse = inverse(a)
    resultat = ""
    for lettre in nettoyer(texte):
        y = ord(lettre) - ord("A")
        x = (a_inverse * (y - b)) % 26
        resultat = resultat + chr(x + ord("A"))
    return resultat


## 1. Essayer avec un message

On peut modifier le message et la clé dans cette cellule. Pour déchiffrer, on utilise l'inverse de a modulo 26 : c'est le nombre qui, multiplié par a, donne un reste égal à 1. Pour a = 5, cet inverse vaut 21, car 5 × 21 = 105 et 105 % 26 = 1.


In [2]:
message = "SALUT ON SE RETROUVE DEMAIN"
a = 5
b = 8

secret = chiffrer(message, a, b)
retrouve = dechiffrer(secret, a, b)

print("Message :", message)
print("Chiffre :", secret)
print("Dechiffre :", retrouve)
print("Message retrouve :", retrouve == nettoyer(message))


Message : SALUT ON SE RETROUVE DEMAIN
,Chiffre : UILEZAVUCPCZPAEJCXCQIWV
,Dechiffre : SALUTONSERETROUVEDEMAIN
,Message retrouve : True


## 2. Vérifier plusieurs messages et plusieurs clés

Un seul exemple ne suffit pas. On vérifie que déchiffrer un message chiffré redonne toujours sa version nettoyée dans les essais ci-dessous. On inclut l'alphabet entier pour tester toutes les lettres, et un message vide.


In [3]:
messages = ["BONJOUR", "RENDEZ VOUS A MIDI", "ABCDEFGHIJKLMNOPQRSTUVWXYZ", ""]
cles = [(1, 3), (5, 8), (7, 12)]
reussites = 0

for message_test in messages:
    for a_test, b_test in cles:
        chiffre = chiffrer(message_test, a_test, b_test)
        clair = dechiffrer(chiffre, a_test, b_test)
        if clair == nettoyer(message_test):
            reussites = reussites + 1

print(reussites, "tests reussis sur", len(messages) * len(cles))


12 tests reussis sur 12


## 3. Pourquoi certaines clés sont refusées

Pour pouvoir déchiffrer, deux lettres différentes ne doivent pas devenir la même lettre. Avec a = 2 et b = 0, A et N deviennent toutes les deux A. On ne peut plus savoir quelle lettre était présente au départ.

Il faut donc que le PGCD de a et 26 soit égal à 1. C'est ce que vérifie le programme.


In [4]:
for lettre in "AN":
    x = ord(lettre) - ord("A")
    y = (2 * x) % 26
    print(lettre, "devient", chr(y + ord("A")))

try:
    chiffrer("SALUT", 2, 0)
except ValueError as erreur:
    print(erreur)


A devient A
,N devient A
,Cle invalide : a doit etre premier avec 26


## 4. Retrouver un message sans la clé

Les valeurs utiles de a et b sont comprises entre 0 et 25 : les autres répètent les mêmes transformations à cause du modulo 26. Il y a 12 valeurs valides de a et 26 valeurs de b, donc seulement **312 clés**.

On essaie toutes ces clés sur le premier message. On suppose que la personne qui cherche connaît un seul mot du message : **SALUT**. On affiche les propositions qui contiennent ce mot. La recherche n'utilise ni la clé choisie ni le reste du message original.


In [5]:
essais = 0
candidats = 0
mot_connu = "SALUT"

for a_test in range(26):
    if pgcd(a_test, 26) == 1:
        for b_test in range(26):
            essais = essais + 1
            proposition = dechiffrer(secret, a_test, b_test)
            if mot_connu in proposition:
                candidats = candidats + 1
                print("Cle possible :", a_test, b_test)
                print("Message possible :", proposition)

print("Nombre de cles essayees :", essais)
print("Nombre de propositions retenues :", candidats)


Cle possible : 5 8
,Message possible : SALUTONSERETROUVEDEMAIN
,Nombre de cles essayees : 312
,Nombre de propositions retenues : 1


## Conclusion

Le programme permet de chiffrer puis de retrouver les lettres d'un message avec une clé valide. Les 12 tests donnent le résultat attendu. La clé a = 2 est refusée, car elle fait perdre de l'information.

Sur le message étudié, les 312 essais retrouvent une seule proposition contenant SALUT : la clé **(5, 8)** et le message initial sans espaces. Ce chiffrement est donc facile à essayer, mais trop faible pour protéger des informations importantes.

La recherche suppose qu'on connaît un mot. Si le mot est absent, elle ne retient rien ; avec un mot très court, elle peut garder plusieurs propositions. Sans ce renseignement, il faudrait lire les 312 résultats ou utiliser l'analyse de fréquences du TP5.

Le programme ne conserve pas les espaces, les accents et la ponctuation. Il utilise uniquement l'alphabet A–Z et des clés entières. Les tests donnent confiance dans les exemples vérifiés, mais ne sont pas une preuve pour toutes les entrées possibles.

## AIDE
IA utliser pour la redaction de texte et une comphrension plus claire .


